In [0]:
%run ./opal_functions


In [0]:
cursor = ""
page_size = 1000
ret = get_opal_users(cursor=cursor, page_size=page_size)
users_accum = ret.json()['results']
if ret.json()['next'] != None:
    cursor = ret.json()['next']
    while cursor != None:
        ret = get_opal_users(cursor=cursor, page_size=page_size)
        users_accum += ret.json()['results']
        if ret.json()['next'] != 'None':
            cursor = ret.json()['next']

In [0]:
import json
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, ArrayType

# Create a SparkSession
spark = SparkSession.builder.appName("StructTypeExample").getOrCreate()

schema = StructType([
    StructField("email", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("hr_idp_status", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("position", StringType(), True),
    StructField("user_id", StringType(), True),
])

df = spark.createDataFrame(users_accum, schema)

# Display the DataFrame
#df.show(1, False)
data_expanded=df.select("email", "first_name", "full_name", "hr_idp_status", "last_name", "position", "user_id")

In [0]:
# output the first row to the screen
df.show(1, False)

+----------------------+------------+------------+-------------+---------+--------+------------------------------------+
|email                 |first_name  |full_name   |hr_idp_status|last_name|position|user_id                             |
+----------------------+------------+------------+-------------+---------+--------+------------------------------------+
|00maniuser_a@gmail.com|00maniuser_a|00maniuser_a|NOT_FOUND    |         |        |c049fcbc-dc23-4b0d-b26e-c24939c4cce3|
+----------------------+------------+------------+-------------+---------+--------+------------------------------------+
only showing top 1 row


In [0]:
df.write.saveAsTable("workspace.opal_dev.opal_demo_users", mode="overwrite") # bronze layer
data_expanded.write.saveAsTable("workspace.opal_dev.opal_demo_users_formatted", mode="overwrite") # silver layer

---------------------------------------------------------------------------
SparkConnectGrpcException                 Traceback (most recent call last)
File <command-3181193331971276>, line 1
----> 1 df.write.saveAsTable("workspace.opal_dev.opal_demo_users", mode="overwrite") # bronze layer
      2 data_expanded.write.saveAsTable("workspace.opal_dev.opal_demo_users_formatted", mode="overwrite")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:713, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    711 self._write.table_name = name
    712 self._write.table_save_method = "save_as_table"
--> 713 _, _, ei = self._spark.client.execute_command(
    714     self._write.command(self._spark.client), self._write.observations
    715 )
    716 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1484, in SparkConnectClient.execute_command(self, command, observations, ex